# httpbin Deployment 空闲生命周期

本 Notebook 独立创建一个 httpbin Deployment，先观察空闲后的 `STOP`，再通过完整配置更新切换到 `PAUSE`。`STOP` 释放 Sandbox Instance；`PAUSE` 保留实例状态以供恢复。httpbin 本身是无状态服务，因此本例观察请求恢复与 Deployment 摘要，不把它当作状态持久性测试。

> ID 和 Token 由读者手工复制。等待过程也由读者计时，不使用轮询脚本。请把 `AGR_ROLE_ARN` 替换为允许 AGR 拉取目标 CCR 镜像的 CAM 角色 ARN。

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-lifecycle-your-name
%env HTTPBIN_DEPLOYMENT_NAME=httpbin-lifecycle-your-name
!agr status

## 1. 创建独立 Tool

先修改名称中的 `your-name`。

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. 配置空闲后停止

复制 `ToolId`。`MinInstanceCount=0` 允许实例在空闲后真正离开活跃容量；空闲 30 秒后，`STOP` 释放实例。

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create \
  --region "$AGR_REGION" \
  --deployment-name "$HTTPBIN_DEPLOYMENT_NAME" \
  --tool-id "$HTTPBIN_TOOL_ID" \
  --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":1,"MaxInstanceRequestConcurrency":10}' \
  --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}'

## 3. 激活实例并观察 `STOP`

复制 `DeploymentId`，获取并复制 `Data.Response.Response.Token`。先请求一次以启动实例，然后保持至少 30 秒不发送请求或连接；之后运行第二个观察单元。实际回收与下一次启动是异步过程，时间可能略长于配置值。

In [ ]:
%env HTTPBIN_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$HTTPBIN_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env HTTPBIN_DEPLOYMENT_TOKEN=dpt-replace-me
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

现在停止发送请求至少 30 秒，再执行下面的查询和请求。`get` 用于查看 Deployment 配置与容量摘要；后续请求会在需要时重新启动实例。

In [ ]:
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 4. 切换为空闲后暂停

生命周期对象在更新时被完整替换，因此同时提供超时与动作。更新后先请求一次，使当前实例在新策略下进入活跃状态；随后再次空闲至少 30 秒，再观察恢复请求。`PAUSE` 的核心语义是保留实例状态，而不是承诺固定恢复延迟。

In [ ]:
!agr deployment update "$HTTPBIN_DEPLOYMENT_ID" \
  --region "$AGR_REGION" \
  --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"PAUSE"}'
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

再次停止发送请求至少 30 秒，然后执行：

In [ ]:
!agr deployment get "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!curl --fail-with-body --silent --show-error --header "X-Access-Token: $HTTPBIN_DEPLOYMENT_TOKEN" "https://8080-$HTTPBIN_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/get"

## 5. 清理资源

In [ ]:
!agr deployment delete "$HTTPBIN_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

`PAUSE` 实验后可能仍能看到暂停实例。复制每个非 `STOPPED` 实例 ID，在新单元先执行 `%env HTTPBIN_INSTANCE_ID=replace-me`，再执行 `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`，然后删除 Tool。

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait